# 04D0 — LANDFIRE Pre-Event Fuel Vintage and Access Audit

**Build ID:** `04D0_LANDFIRE_PRE_EVENT_VINTAGE_AUDIT_V1`

This notebook does **not** extract fuel pixels yet. It freezes the temporally admissible LANDFIRE vintage for each target-fire year and checks whether public ArcGIS Online candidates exist for the five required products.

Primary temporal rule:

\[
	ext{max disturbance year} < 	ext{fire year}
\]

and

\[
	ext{fuel effective/capable year} \le 	ext{fire year}
\]

This prevents same-fire disturbance leakage and prevents use of a future capable-fuel state.

Candidate mapping expected from that rule:

- 2020 → LF 2016 Remap / 2020-capable fuels
- 2021 → LF 2019 Limited / 2021-capable fuels
- 2022 → LF 2020 / 2022-capable fuels
- 2023 → LF 2022
- 2024 → LF 2023

Important: this is a **retrospective pre-event landscape-state reconstruction**, not automatically proof that the exact product was publicly downloadable at the fire date.

Products audited: `FBFM40`, `CC`, `CH`, `CBH`, `CBD`.

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04D0_LANDFIRE_PRE_EVENT_VINTAGE_AUDIT_V1"

from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

TERRAIN_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "terrain_3dep_features_v1.parquet"
)

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "landfire_vintage_audit_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "landfire_vintage_audit_v1"

for p in [OUTPUT_ROOT, PROCESSED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

assert TERRAIN_PATH.exists(), TERRAIN_PATH

PRODUCTS = ["FBFM40", "CC", "CH", "CBH", "CBD"]

ARCGIS_SEARCH_URL = "https://www.arcgis.com/sharing/rest/search"
REQUEST_TIMEOUT_S = 45

print("RUNNING:", NOTEBOOK_BUILD_ID)
print("Terrain input:", TERRAIN_PATH)

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "requests": "requests",
    "pyarrow": "pyarrow",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from datetime import datetime, timezone
import time
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 320)

## 1. Frozen terrain cohort

In [ ]:
terrain = pd.read_parquet(TERRAIN_PATH)

assert terrain["unit_uid"].is_unique

terrain["Year"] = pd.to_numeric(
    terrain["Year"], errors="raise"
).astype(int)

print("Units:", f"{len(terrain):,}")
print("Incidents:", terrain["_irwin_norm"].nunique())

year_summary = (
    terrain.groupby("Year", as_index=False)
    .agg(
        units=("unit_uid", "size"),
        incidents=("_irwin_norm", "nunique"),
        burned_over=(
            "_engagement_norm",
            lambda s: (s == "Burned Over").sum(),
        ),
        held=(
            "_engagement_norm",
            lambda s: (s == "Held").sum(),
        ),
    )
)

display(year_summary)

## 2. LANDFIRE version catalog used for temporal selection

In [ ]:
version_catalog = pd.DataFrame([
    {
        "version_key": "LF2014",
        "version_name": "LF 2014",
        "version_number": "1.4.x",
        "max_disturbance_year": 2014,
        "fuel_effective_year": 2014,
        "completion_year": 2017,
        "limited_disturbance": False,
        "archived": True,
        "note": "Older fallback only.",
    },
    {
        "version_key": "LF2016R",
        "version_name": "LF 2016 Remap",
        "version_number": "2.0.0",
        "max_disturbance_year": 2016,
        "fuel_effective_year": 2020,
        "completion_year": 2021,
        "limited_disturbance": False,
        "archived": False,
        "note": "New base map; capable fuels transitioned to expected 2020 conditions.",
    },
    {
        "version_key": "LF2019L",
        "version_name": "LF 2019 Limited",
        "version_number": "2.1.x",
        "max_disturbance_year": 2019,
        "fuel_effective_year": 2021,
        "completion_year": 2021,
        "limited_disturbance": True,
        "archived": True,
        "note": "2021-capable; 2017–2019 disturbance mapping was limited.",
    },
    {
        "version_key": "LF2020",
        "version_name": "LF 2020",
        "version_number": "2.2.0",
        "max_disturbance_year": 2020,
        "fuel_effective_year": 2022,
        "completion_year": 2023,
        "limited_disturbance": False,
        "archived": True,
        "note": "2022-capable; full disturbance mapping through 2020.",
    },
    {
        "version_key": "LF2022",
        "version_name": "LF 2022",
        "version_number": "2.3.0",
        "max_disturbance_year": 2022,
        "fuel_effective_year": 2022,
        "completion_year": 2023,
        "limited_disturbance": False,
        "archived": False,
        "note": "Includes 2021–2022 disturbance.",
    },
    {
        "version_key": "LF2023",
        "version_name": "LF 2023",
        "version_number": "2.4.0",
        "max_disturbance_year": 2023,
        "fuel_effective_year": 2023,
        "completion_year": 2024,
        "limited_disturbance": False,
        "archived": False,
        "note": "Includes disturbance through 2023.",
    },
    {
        "version_key": "LF2024",
        "version_name": "LF 2024",
        "version_number": "2.5.0",
        "max_disturbance_year": 2024,
        "fuel_effective_year": 2024,
        "completion_year": 2025,
        "limited_disturbance": False,
        "archived": False,
        "note": "Contains target-year disturbance for 2024 fires; unsafe for primary 2024 target modeling.",
    },
])

display(version_catalog)

## 3. Automatically derive the pre-event version mapping

In [ ]:
def select_pre_event_version(fire_year):
    admissible = version_catalog[
        (version_catalog["max_disturbance_year"] < fire_year)
        & (version_catalog["fuel_effective_year"] <= fire_year)
    ].copy()

    if admissible.empty:
        return None

    return (
        admissible.sort_values(
            ["max_disturbance_year", "fuel_effective_year"],
            ascending=[False, False],
        )
        .iloc[0]
    )

rows = []

for year in sorted(terrain["Year"].unique()):
    sel = select_pre_event_version(int(year))
    if sel is None:
        raise RuntimeError(f"No admissible LANDFIRE version for {year}")

    rows.append({
        "fire_year": int(year),
        "version_key": sel["version_key"],
        "version_name": sel["version_name"],
        "version_number": sel["version_number"],
        "max_disturbance_year": int(sel["max_disturbance_year"]),
        "fuel_effective_year": int(sel["fuel_effective_year"]),
        "completion_year": int(sel["completion_year"]),
        "limited_disturbance": bool(sel["limited_disturbance"]),
        "archived": bool(sel["archived"]),
        "pre_event_disturbance_safe": bool(sel["max_disturbance_year"] < year),
        "future_capable_state_safe": bool(sel["fuel_effective_year"] <= year),
        "strictly_completed_before_fire_year": bool(sel["completion_year"] < year),
        "note": sel["note"],
    })

vintage_mapping = pd.DataFrame(rows)

expected = {
    2020: "LF2016R",
    2021: "LF2019L",
    2022: "LF2020",
    2023: "LF2022",
    2024: "LF2023",
}

observed = dict(zip(vintage_mapping["fire_year"], vintage_mapping["version_key"]))

assert observed == expected, observed
assert vintage_mapping["pre_event_disturbance_safe"].all()
assert vintage_mapping["future_capable_state_safe"].all()

display(vintage_mapping)

print("PRE-EVENT TEMPORAL RULE: PASS")

## 4. Assign the vintage to every operational unit

In [ ]:
unit_vintage = (
    terrain[
        ["unit_uid", "_irwin_norm", "IncidentName", "Year", "_engagement_norm"]
    ]
    .merge(
        vintage_mapping,
        left_on="Year",
        right_on="fire_year",
        how="left",
        validate="many_to_one",
    )
)

assert len(unit_vintage) == len(terrain)
assert unit_vintage["version_key"].notna().all()

vintage_cohort_summary = (
    unit_vintage.groupby(
        ["fire_year", "version_key", "version_name", "limited_disturbance", "archived"],
        as_index=False,
    )
    .agg(
        units=("unit_uid", "size"),
        incidents=("_irwin_norm", "nunique"),
        burned_over=(
            "_engagement_norm",
            lambda s: (s == "Burned Over").sum(),
        ),
        held=(
            "_engagement_norm",
            lambda s: (s == "Held").sum(),
        ),
    )
)

display(vintage_cohort_summary)

UNIT_VINTAGE_PATH = (
    PROCESSED_ROOT / "unit_landfire_vintage_assignment_v1.parquet"
)

unit_vintage.to_parquet(UNIT_VINTAGE_PATH, index=False)

vintage_mapping.to_csv(
    OUTPUT_ROOT / "LANDFIRE_PRE_EVENT_VINTAGE_MAPPING.csv",
    index=False,
)

## 5. Product semantics to carry into 04D1

In [ ]:
product_semantics = pd.DataFrame([
    {
        "product": "FBFM40",
        "kind": "categorical",
        "conversion": "none",
        "planned_summary": "dominant model; dominant fraction; entropy; verified fuel-family fractions",
    },
    {
        "product": "CC",
        "kind": "canopy cover",
        "conversion": "percent",
        "planned_summary": "mean; p90; max; nonzero/forest fraction",
    },
    {
        "product": "CH",
        "kind": "canopy height",
        "conversion": "raw / 10 = meters",
        "planned_summary": "mean; p90; max; nonzero/forest fraction",
    },
    {
        "product": "CBH",
        "kind": "canopy base height",
        "conversion": "raw / 10 = meters",
        "planned_summary": "mean; p10; median; p90 among forest pixels",
    },
    {
        "product": "CBD",
        "kind": "canopy bulk density",
        "conversion": "raw / 100 = kg/m^3",
        "planned_summary": "mean; p90; max among forest pixels",
    },
])

display(product_semantics)

product_semantics.to_csv(
    OUTPUT_ROOT / "LANDFIRE_PRODUCT_SEMANTICS.csv",
    index=False,
)

## 6. ArcGIS Online historical-service discovery

This is deliberately a discovery audit. It does not trust or freeze a service simply because a title matches.

The notebook searches the public ArcGIS Online catalog for every required `version × product` combination, then records the top candidates for review.

In [ ]:
VERSION_TERMS = {
    "LF2016R": ["LANDFIRE 2016 Remap", "LANDFIRE 2016", "200"],
    "LF2019L": ["LANDFIRE 2019 Limited", "LANDFIRE 2019", "2019L"],
    "LF2020": ["LANDFIRE 2020", "220"],
    "LF2022": ["LANDFIRE 2022", "230"],
    "LF2023": ["LANDFIRE 2023", "240"],
}

PRODUCT_TERMS = {
    "FBFM40": ["FBFM40", "Scott", "F40"],
    "CC": ["Canopy Cover", " CC "],
    "CH": ["Canopy Height", " CH "],
    "CBH": ["Canopy Base Height", "CBH"],
    "CBD": ["Canopy Bulk Density", "CBD"],
}

def agol_search(query):
    r = requests.get(
        ARCGIS_SEARCH_URL,
        params={"q": query, "f": "json", "num": 100},
        timeout=REQUEST_TIMEOUT_S,
    )
    r.raise_for_status()
    payload = r.json()
    if "error" in payload:
        raise RuntimeError(payload["error"])
    return payload.get("results", [])

def score_candidate(title, url, version_key, product):
    text = f"{title or ''} {url or ''}".lower()
    score = 0

    for term in VERSION_TERMS[version_key]:
        if term.lower() in text:
            score += 4

    for term in PRODUCT_TERMS[product]:
        if term.lower() in text:
            score += 5

    if "landfire" in text:
        score += 3
    if "conus" in text:
        score += 2

    return score

candidate_rows = []

required_versions = vintage_mapping["version_key"].drop_duplicates().tolist()

for version_key in required_versions:
    version_name = (
        version_catalog.loc[
            version_catalog["version_key"] == version_key,
            "version_name",
        ]
        .iloc[0]
    )

    for product in PRODUCTS:
        queries = [
            f'"{version_name}" {product}',
            f'LANDFIRE {version_name.replace("LF ", "")} {product}',
        ]

        collected = {}

        for q in queries:
            try:
                results = agol_search(q)
            except Exception as exc:
                candidate_rows.append({
                    "version_key": version_key,
                    "version_name": version_name,
                    "product": product,
                    "candidate_rank": np.nan,
                    "score": np.nan,
                    "item_id": pd.NA,
                    "title": pd.NA,
                    "type": pd.NA,
                    "owner": pd.NA,
                    "service_url": pd.NA,
                    "search_error": f"{type(exc).__name__}: {exc}",
                })
                continue

            for item in results:
                if item.get("id"):
                    collected[item["id"]] = item

        ranked = sorted(
            [
                (
                    score_candidate(
                        item.get("title"),
                        item.get("url"),
                        version_key,
                        product,
                    ),
                    item,
                )
                for item in collected.values()
            ],
            key=lambda x: x[0],
            reverse=True,
        )

        for rank, (score, item) in enumerate(ranked[:10], start=1):
            candidate_rows.append({
                "version_key": version_key,
                "version_name": version_name,
                "product": product,
                "candidate_rank": rank,
                "score": score,
                "item_id": item.get("id"),
                "title": item.get("title"),
                "type": item.get("type"),
                "owner": item.get("owner"),
                "service_url": item.get("url"),
                "search_error": pd.NA,
            })

        time.sleep(0.1)

agol_candidates = pd.DataFrame(candidate_rows)

display(
    agol_candidates[
        agol_candidates["item_id"].notna()
    ].head(100)
)

agol_candidates.to_csv(
    OUTPUT_ROOT / "LANDFIRE_AGOL_SEARCH_CANDIDATES.csv",
    index=False,
)

## 7. Validate the highest-ranked candidate service per version × product

In [ ]:
usable = agol_candidates[
    agol_candidates["item_id"].notna()
].copy()

if len(usable):
    best = (
        usable.sort_values(
            ["version_key", "product", "score", "candidate_rank"],
            ascending=[True, True, False, True],
        )
        .groupby(["version_key", "product"], as_index=False)
        .first()
    )
else:
    best = pd.DataFrame(
        columns=[
            "version_key", "version_name", "product", "item_id",
            "title", "type", "owner", "service_url", "score",
        ]
    )

def inspect_service(url):
    if not isinstance(url, str) or not url.strip():
        return {
            "service_http_ok": False,
            "service_error": "missing_service_url",
        }

    try:
        r = requests.get(
            url,
            params={"f": "pjson"},
            timeout=REQUEST_TIMEOUT_S,
        )
        r.raise_for_status()
        payload = r.json()

        if "error" in payload:
            return {
                "service_http_ok": False,
                "service_error": str(payload["error"]),
            }

        extent = payload.get("extent") or {}
        sr = extent.get("spatialReference") or {}

        return {
            "service_http_ok": True,
            "service_name": payload.get("name"),
            "pixel_type": payload.get("pixelType"),
            "band_count": payload.get("bandCount"),
            "wkid": sr.get("latestWkid") or sr.get("wkid"),
            "service_error": pd.NA,
        }

    except Exception as exc:
        return {
            "service_http_ok": False,
            "service_error": f"{type(exc).__name__}: {exc}",
        }

service_rows = []

for rec in best.to_dict("records"):
    service_rows.append({
        **rec,
        **inspect_service(rec.get("service_url")),
    })
    time.sleep(0.1)

service_audit = pd.DataFrame(service_rows)

display(service_audit)

service_audit.to_csv(
    OUTPUT_ROOT / "LANDFIRE_AGOL_SERVICE_AUDIT.csv",
    index=False,
)

## 8. Decision table and final report

In [ ]:
decision_rows = []

for rec in vintage_mapping.to_dict("records"):
    subset = service_audit[
        service_audit["version_key"] == rec["version_key"]
    ]

    validated = set(
        subset.loc[
            subset["service_http_ok"] == True,
            "product",
        ]
    )

    decision_rows.append({
        "fire_year": rec["fire_year"],
        "version_key": rec["version_key"],
        "version_name": rec["version_name"],
        "max_disturbance_year": rec["max_disturbance_year"],
        "fuel_effective_year": rec["fuel_effective_year"],
        "limited_disturbance": rec["limited_disturbance"],
        "archived": rec["archived"],
        "temporally_safe": (
            rec["pre_event_disturbance_safe"]
            and rec["future_capable_state_safe"]
        ),
        "validated_product_count": len(validated),
        "validated_products": ";".join(sorted(validated)),
        "all_5_products_validated": set(PRODUCTS).issubset(validated),
        "strictly_completed_before_fire_year": rec["strictly_completed_before_fire_year"],
    })

decision_table = pd.DataFrame(decision_rows)

checks = pd.DataFrame([
    {
        "check": "all_fire_years_temporally_safe",
        "pass": bool(decision_table["temporally_safe"].all()),
    },
    {
        "check": "no_target_year_disturbance_in_selected_version",
        "pass": bool(
            (
                vintage_mapping["max_disturbance_year"]
                < vintage_mapping["fire_year"]
            ).all()
        ),
    },
    {
        "check": "no_future_capable_fuel_state",
        "pass": bool(
            (
                vintage_mapping["fuel_effective_year"]
                <= vintage_mapping["fire_year"]
            ).all()
        ),
    },
    {
        "check": "all_required_AGOL_product_services_validated",
        "pass": bool(
            len(decision_table)
            and decision_table["all_5_products_validated"].all()
        ),
    },
])

display(decision_table)
display(checks)

decision_table.to_csv(
    OUTPUT_ROOT / "LANDFIRE_VINTAGE_ACCESS_DECISION_TABLE.csv",
    index=False,
)

checks.to_csv(
    OUTPUT_ROOT / "LANDFIRE_VINTAGE_AUDIT_CHECKS.csv",
    index=False,
)

def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report = f"""# 04D0 LANDFIRE Pre-Event Vintage and Access Audit

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Cohort

Units: **{len(terrain):,}**

Incidents: **{terrain["_irwin_norm"].nunique():,}**

## Temporal rule

A selected fuel version must satisfy:

`max disturbance year < target fire year`

and:

`fuel effective/capable year <= target fire year`

## Candidate mapping

{tomd(vintage_mapping)}

## Cohort by vintage

{tomd(vintage_cohort_summary)}

## Product semantics

{tomd(product_semantics)}

## Access decision

{tomd(decision_table)}

## Checks

{tomd(checks)}

## Provenance interpretation

The selected version is a **retrospectively reconstructed pre-event LANDFIRE fuel state**.

It must not automatically be described as the exact product operationally available at the fire date unless a separate release-date audit proves that.

## LF 2019 Limited caveat

2021 fires map to LF 2019 Limited because it is the newest version satisfying both temporal constraints.

Its 2017–2019 disturbance mapping is limited, so retain `limited_disturbance=True` as a vintage-quality flag and test sensitivity by year/vintage later.

## Next

If every required historical product service is validated, proceed to:

`04D1_LANDFIRE_fuel_features.ipynb`

If not, review:

- `LANDFIRE_AGOL_SEARCH_CANDIDATES.csv`
- `LANDFIRE_AGOL_SERVICE_AUDIT.csv`

and resolve archived services before fuel extraction.
"""

REPORT_PATH = (
    OUTPUT_ROOT / "LANDFIRE_PRE_EVENT_VINTAGE_AUDIT_REPORT.md"
)

REPORT_PATH.write_text(report, encoding="utf-8")

print(report)
print()
print("Saved:", REPORT_PATH)

## 9. Output inventory

In [ ]:
print("=" * 100)
print("04D0 OUTPUTS")
print("=" * 100)

print("Unit vintage assignment:")
print(UNIT_VINTAGE_PATH)

print()
print("Report:")
print(REPORT_PATH)

print()
for p in sorted(OUTPUT_ROOT.glob("*")):
    if p.is_file():
        print(
            f"{p.name:<75}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print("DONE:", NOTEBOOK_BUILD_ID)